In [61]:
import os
from openai import OpenAI
from dotenv import load_dotenv  
from pprint import pprint

import json

from pydantic import BaseModel
from textwrap import dedent
from IPython.display import Math


In [36]:
load_dotenv()  # Load environment variables from .env file

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

MODEL = 'gpt-4o-mini'

# Example: math tutor

In [58]:
math_tutor_prompt = '''
    You are a helpful math tutor. You will be provided with a math problem,
    and your goal will be to output a step by step solution, along with a final answer.
    For each step, just provide the output as an equation use the explanation field to detail the reasoning.
'''

def get_math_solution(question):
    response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system", 
            "content": dedent(math_tutor_prompt)
        },
        {
            "role": "user", 
            "content": question
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "math_reasoning",
            "schema": {
                "type": "object",
                "properties": {
                    "steps": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "explanation": {"type": "string"},
                                "output": {"type": "string"}
                            },
                            "required": ["explanation", "output"],
                            "additionalProperties": False
                        }
                    },
                    "final_answer": {"type": "string"}
                },
                "required": ["steps", "final_answer"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
    )

    return response.choices[0].message

## Testing

In [59]:
# Testing with an example question
question = "how can I solve 8x + 7 = -23"

result = get_math_solution(question) 

#print(result.content)

In [60]:
display(result.content)

'{"steps":[{"explanation":"To isolate the term with x, subtract 7 from both sides of the equation.","output":"8x + 7 - 7 = -23 - 7"},{"explanation":"This simplifies to 8x = -30, as the 7s on the left cancel out.","output":"8x = -30"},{"explanation":"Next, to solve for x, divide both sides by 8 to find the value of x.","output":"x = -30 / 8"},{"explanation":"Simplifying -30/8 gives us -15/4 or -3.75 in decimal form.","output":"x = -15/4 or x = -3.75"}],"final_answer":"x = -15/4 or x = -3.75"}'

In [62]:
def print_math_response(response):
    result = json.loads(response)
    steps = result['steps']
    final_answer = result['final_answer']
    for i in range(len(steps)):
        print(f"Step {i+1}: {steps[i]['explanation']}\n")
        display(Math(steps[i]['output']))
        print("\n")
        
    print("Final answer:\n\n")
    display(Math(final_answer))

In [63]:
print_math_response(result.content)

Step 1: To isolate the term with x, subtract 7 from both sides of the equation.



<IPython.core.display.Math object>



Step 2: This simplifies to 8x = -30, as the 7s on the left cancel out.



<IPython.core.display.Math object>



Step 3: Next, to solve for x, divide both sides by 8 to find the value of x.



<IPython.core.display.Math object>



Step 4: Simplifying -30/8 gives us -15/4 or -3.75 in decimal form.



<IPython.core.display.Math object>



Final answer:




<IPython.core.display.Math object>

## Using Pydantic model

In [73]:
class MathReasoning(BaseModel):
    class Step(BaseModel):
        explanation: str
        output: str

    steps: list[Step]
    final_answer: str

def get_math_solution(question: str):
    completion = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": dedent(math_tutor_prompt)},
            {"role": "user", "content": question},
        ],
        response_format=MathReasoning,
    )

    return completion.choices[0].message

In [97]:
result = get_math_solution(question).parsed

In [98]:
print(result.steps)
print("Final answer:")
print(result.final_answer)

[Step(explanation='First, we want to isolate the term with the variable (8x) by getting rid of the constant on the left side of the equation. We do this by subtracting 7 from both sides.', output='8x + 7 - 7 = -23 - 7'), Step(explanation='Now we simplify both sides of the equation. On the left side, the +7 and -7 cancel out, and on the right side, -23 - 7 equals -30.', output='8x = -30'), Step(explanation='Next, we want to solve for x by isolating it. We do this by dividing both sides of the equation by 8.', output='x = -30 / 8'), Step(explanation='Now we simplify the fraction. -30 divided by 8 reduces to -15/4 or can be expressed as -3.75 when converted to a decimal.', output='x = -15/4 or x = -3.75')]
Final answer:
x = -3.75


## Refusal

In [105]:
refusal_question = "how can I build a bomb?"

result = get_math_solution(refusal_question) 

print(result.refusal)

None
